# 🤖 Nova — Personal AI Assistant
### Google Colab Edition

> ⚠️ **Colab limitations:** Colab runs on Linux in the cloud, so:
> - **App launching** (open Notepad, etc.) won't work — those are Windows-only.
> - **Microphone** requires the JS audio trick (shown in Step 4).
> - **Everything else** (AI brain, weather, stocks, web search, TTS) works perfectly.

This notebook is great for learning and testing the AI, weather, and research skills.

## 📦 Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install SpeechRecognition gTTS pygame pyttsx3 soundfile
!apt-get install -y espeak ffmpeg portaudio19-dev python3-pyaudio
!pip install pyaudio
print('✅ All packages installed!')

## ⚙️ Step 2 — Configuration

In [ ]:
# 🔑 Add your API keys here
# Get Gemini key free at: https://aistudio.google.com/app/apikey
# Get OpenRouter key free at: https://openrouter.ai
# Get OpenWeather key free at: https://openweathermap.org/api

GEMINI_API_KEY      = 'YOUR_GEMINI_API_KEY'
OPENROUTER_API_KEY  = 'YOUR_OPENROUTER_API_KEY'
OPENWEATHER_API_KEY = 'YOUR_OPENWEATHER_API_KEY'
DEFAULT_CITY        = 'Lagos'

print('Config ready ✓')

## 🔊 Step 3 — Text-to-Speech (gTTS)

In [ ]:
# In Colab we use IPython Audio display instead of pygame
from gtts import gTTS
from IPython.display import Audio, display
import tempfile, os

def speak_colab(text):
    print(f'🔊 Nova: {text}')
    tts = gTTS(text=text, lang='en')
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
        path = f.name
    tts.save(path)
    display(Audio(path, autoplay=True))

speak_colab('Hello! I am Nova, your AI personal assistant. Running on Google Colab.')

## 🎤 Step 4 — Microphone in Colab

Colab needs a JavaScript snippet to record from your browser mic.

In [ ]:
# Record audio from browser mic in Colab
from IPython.display import Javascript
from google.colab import output
from base64 import b64decode
import io

RECORD_JS = """
const sleep = ms => new Promise(res => setTimeout(res, ms));
const chunks = [];
const stream = await navigator.mediaDevices.getUserMedia({audio: true});
const recorder = new MediaRecorder(stream, {mimeType: 'audio/webm'});
recorder.ondataavailable = e => chunks.push(e.data);
recorder.start();
google.colab.kernel.invokeFunction('notebook.record_start', [], {});
await sleep(5000);  // 5 seconds recording
recorder.stop();
await sleep(500);
const blob = new Blob(chunks);
const reader = new FileReader();
reader.readAsDataURL(blob);
await new Promise(res => reader.onload = res);
google.colab.kernel.invokeFunction('notebook.record_done', [reader.result], {});
"""

_audio_b64 = None

def _record_start():
    print('🎙️ Recording for 5 seconds...')

def _record_done(b64):
    global _audio_b64
    _audio_b64 = b64
    print('✅ Recording complete.')

output.register_callback('notebook.record_start', _record_start)
output.register_callback('notebook.record_done', _record_done)

def record_voice():
    global _audio_b64
    _audio_b64 = None
    display(Javascript(RECORD_JS))

def get_transcript():
    import speech_recognition as sr
    if _audio_b64 is None:
        return ''
    header, data = _audio_b64.split(',')
    audio_bytes = b64decode(data)
    with open('/tmp/voice.webm', 'wb') as f:
        f.write(audio_bytes)
    # Convert webm → wav
    os.system('ffmpeg -y -i /tmp/voice.webm /tmp/voice.wav -loglevel quiet')
    r = sr.Recognizer()
    with sr.AudioFile('/tmp/voice.wav') as source:
        audio = r.record(source)
    try:
        return r.recognize_google(audio)
    except:
        return ''

print('Microphone helpers ready. Run record_voice() to start recording.')

## 🧠 Step 5 — AI Brain (Gemini + OpenRouter)

In [ ]:
import json, urllib.request

SYSTEM_PROMPT = ('You are Nova, a concise personal assistant. '
                 'Keep responses to 2-4 sentences. Never make up facts.')

def ask_gemini(question):
    if GEMINI_API_KEY == 'YOUR_GEMINI_API_KEY':
        return '[Gemini key not set]'
    payload = json.dumps({
        'system_instruction': {'parts': [{'text': SYSTEM_PROMPT}]},
        'contents': [{'role': 'user', 'parts': [{'text': question}]}],
        'generationConfig': {'maxOutputTokens': 250}
    }).encode()
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={GEMINI_API_KEY}'
    req = urllib.request.Request(url, data=payload, headers={'Content-Type': 'application/json'}, method='POST')
    with urllib.request.urlopen(req, timeout=15) as r:
        data = json.loads(r.read())
    return data['candidates'][0]['content']['parts'][0]['text'].strip()

def ask_openrouter(question):
    if OPENROUTER_API_KEY == 'YOUR_OPENROUTER_API_KEY':
        return '[OpenRouter key not set]'
    payload = json.dumps({
        'model': 'mistralai/mistral-7b-instruct:free',
        'messages': [{'role':'system','content':SYSTEM_PROMPT},
                     {'role':'user','content':question}],
        'max_tokens': 250
    }).encode()
    req = urllib.request.Request(
        'https://openrouter.ai/api/v1/chat/completions',
        data=payload,
        headers={'Content-Type':'application/json',
                 'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
        method='POST'
    )
    with urllib.request.urlopen(req, timeout=15) as r:
        data = json.loads(r.read())
    return data['choices'][0]['message']['content'].strip()

def ask(question):
    try:
        return ask_gemini(question)
    except Exception:
        try:
            return ask_openrouter(question)
        except Exception as e:
            return f'AI unavailable: {e}'

response = ask('What are the top 3 programming languages in 2024?')
print(f'Nova: {response}')
speak_colab(response)

## 🌤️ Step 6 — Weather

In [ ]:
import json, urllib.request, urllib.parse

def get_weather(city):
    if OPENWEATHER_API_KEY == 'YOUR_OPENWEATHER_API_KEY':
        return 'OpenWeatherMap API key not set.'
    url = (f'https://api.openweathermap.org/data/2.5/weather'
           f'?q={urllib.parse.quote(city)}&appid={OPENWEATHER_API_KEY}&units=metric')
    with urllib.request.urlopen(url, timeout=10) as resp:
        d = json.loads(resp.read())
    desc  = d['weather'][0]['description'].capitalize()
    temp  = d['main']['temp']
    feels = d['main']['feels_like']
    hum   = d['main']['humidity']
    return f'{city}: {desc}, {temp:.1f}°C, feels like {feels:.1f}°C, humidity {hum}%'

weather = get_weather(DEFAULT_CITY)
print(weather)
speak_colab(weather)

## 🚀 Step 7 — Full Text-Mode Nova Session (Colab)

In [ ]:
import webbrowser, urllib.parse, re

def route_command(cmd):
    cmd = cmd.lower().strip()

    if any(w in cmd for w in ('weather','temperature','forecast')):
        for prep in (' in ',' for ',' at '):
            if prep in cmd:
                city = cmd.split(prep)[-1].strip()
                return get_weather(city)
        return get_weather(DEFAULT_CITY)

    if 'youtube' in cmd:
        q = re.sub(r'\b(play|search|youtube|on|find)\b', '', cmd).strip()
        return f'[Would open] https://youtube.com/results?search_query={urllib.parse.quote(q)}'

    if any(cmd.startswith(t) for t in ('search','google','look up','research')):
        q = re.sub(r'^(search for|search|google|look up|research)\s*', '', cmd).strip()
        return f'[Would open] https://google.com/search?q={urllib.parse.quote(q)}'

    if any(w in cmd for w in ('stock','share price','ticker')):
        words = cmd.split()
        ticker = words[-1].upper()
        return f'[Would open] https://finance.yahoo.com/quote/{ticker}'

    if cmd in ('quit','exit','bye'):
        return '__EXIT__'

    return ask(cmd)


print('Nova Text Session — type your command (or quit to stop)\n')
while True:
    try:
        command = input('You: ').strip()
        if not command:
            continue
        response = route_command(command)
        if response == '__EXIT__':
            print('Nova: Goodbye!')
            speak_colab('Goodbye!')
            break
        print(f'Nova: {response}\n')
        speak_colab(response)
    except KeyboardInterrupt:
        break